In [1]:
import requests

print(requests.get(
    "http://nhits-model-vanilla.lstm-iqu.svc.cluster.local/v2/health/ready",
    timeout=30
).status_code)

200


In [5]:
import requests
import pandas as pd
import json

service_name = "pip-nhits-model-vanilla"
namespace = "lstm-iqu"

infer_url = f"http://{service_name}.{namespace}.svc.cluster.local/v2/models/{service_name}/infer"

test_df = pd.DataFrame({
    "time": pd.date_range("2026-03-24 00:00:00", periods=24, freq="h"),
    "value": [
        0.42, 0.40, 0.39, 0.41, 0.43, 0.45,
        0.44, 0.46, 0.48, 0.47, 0.49, 0.50,
        0.52, 0.51, 0.53, 0.55, 0.54, 0.56,
        0.57, 0.58, 0.56, 0.55, 0.54, 0.53,
    ]
})

payload = {
    "parameters": {"content_type": "pd"},
    "inputs": [
        {
            "name": "time",
            "shape": [len(test_df), 1],
            "datatype": "BYTES",
            "parameters": {"content_type": "str"},
            "data": test_df["time"].dt.strftime("%Y-%m-%dT%H:%M:%S").tolist(),
        },
        {
            "name": "value",
            "shape": [len(test_df), 1],
            "datatype": "FP64",
            "data": test_df["value"].astype(float).tolist(),
        },
    ],
}

response = requests.post(infer_url, json=payload, timeout=120)

print("Status:", response.status_code)
print(response.text)

if response.ok:
    body = response.json()
    print(json.dumps(body, indent=2))


Status: 200
{"model_name":"pip-nhits-model-vanilla","id":"4698f5d4-71eb-4100-a04c-4bf9a6a8a801","parameters":{"content_type":"pd"},"outputs":[{"name":"time","shape":[12,1],"datatype":"BYTES","parameters":{"content_type":"str"},"data":["2026-03-25 00:00:00","2026-03-25 01:00:00","2026-03-25 02:00:00","2026-03-25 03:00:00","2026-03-25 04:00:00","2026-03-25 05:00:00","2026-03-25 06:00:00","2026-03-25 07:00:00","2026-03-25 08:00:00","2026-03-25 09:00:00","2026-03-25 10:00:00","2026-03-25 11:00:00"]},{"name":"value","shape":[12,1],"datatype":"FP64","data":[0.6622426447763025,0.7844097583255258,0.8728096127024478,0.8708237005818945,0.8775125084503576,0.883353626407499,0.9823193800265092,1.0776200173436465,1.0856843757211299,0.9682503548090803,0.7967898053072879,0.6722799261158464]}]}
{
  "model_name": "pip-nhits-model-vanilla",
  "id": "4698f5d4-71eb-4100-a04c-4bf9a6a8a801",
  "parameters": {
    "content_type": "pd"
  },
  "outputs": [
    {
      "name": "time",
      "shape": [
        12